# Sprint 1 — BluaDiagnostics

## IA Conversacional para Check-up Digital

PoC acadêmica desenvolvida para o Challenge BluaDiagnostics utilizando:

- Prompt Engineering;
- memória conversacional;
- function calling simulado;
- integração com Ollama Cloud.

O notebook demonstra:

- system prompt clínico;
- guardrails;
- tools simuladas;
- fluxo de suporte ao beneficiário;
- contexto multi-turno.


## 1. Instalação das dependências

In [8]:
!pip install -q ollama ipython

'pip' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


## 2. Configuração do Ollama Cloud

No menu lateral esquerdo do Colab:

🔑 Secrets

Crie:

```text
OLLAMA_API_KEY
```

Cole sua chave do Ollama Cloud.


In [9]:
from dotenv import load_dotenv
import os
from ollama import Client

load_dotenv()

OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

if not OLLAMA_API_KEY:
    raise ValueError("Configure OLLAMA_API_KEY no arquivo .env")

client = Client(
    host="https://ollama.com",
    headers={
        "Authorization": "Bearer " + OLLAMA_API_KEY
    }
)

MODEL_NAME = "gpt-oss:120b"

print("✅ Ollama Cloud configurado com sucesso")

✅ Ollama Cloud configurado com sucesso


## 3. Teste de conexão

In [10]:
teste = client.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Responda apenas OK"
        }
    ],
    stream=False
)

print("Resposta do modelo:")
print(teste["message"]["content"])


Resposta do modelo:
OK


## 4. System Prompt Clínico

O agente atua como assistente de apoio ao beneficiário da Care Plus.


In [11]:
system_prompt = """
Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Respeite LGPD e minimize coleta de dados sensíveis.

ESCALADA HUMANA:
Encaminhe imediatamente situações de:
- falta de ar intensa;
- dor no peito;
- desmaio;
- confusão mental;
- sinais neurológicos;
- sangramentos importantes.

FORMATO:
1. Resumo do caso
2. Pontos de atenção
3. Próxima ação
4. Aviso médico
"""

print(system_prompt)



Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Respeite LGPD e minimize coleta de dados sensíveis.

ESCALADA HUMANA:
Encaminhe imediatamente situações de:
- falta de ar intensa;
- dor no peito;
- desmaio;
- confusão mental;
- sinais neurológicos;
- sangramentos importantes.

FORMATO:
1. Resumo do caso
2. Pontos de atenção
3. Próxima ação
4. Aviso médico



## 5. Dados simulados do paciente

In [12]:
import json

paciente_mock = {
    "paciente_id": "PAC001",
    "nome": "Paciente Simulado",
    "idade": 42,
    "condicoes": ["hipertensão leve"],
    "medicamentos": ["losartana 50mg"],
    "alergias": ["dipirona"],
    "ultima_pressao": "135/85",
    "ultima_teleconsulta": "2026-05-10"
}

print(json.dumps(
    paciente_mock,
    indent=2,
    ensure_ascii=False
))


{
  "paciente_id": "PAC001",
  "nome": "Paciente Simulado",
  "idade": 42,
  "condicoes": [
    "hipertensão leve"
  ],
  "medicamentos": [
    "losartana 50mg"
  ],
  "alergias": [
    "dipirona"
  ],
  "ultima_pressao": "135/85",
  "ultima_teleconsulta": "2026-05-10"
}


## 6. Function Calling Simulado

In [13]:
def consultar_historico_paciente(paciente_id):
    print("🔧 Executando: consultar_historico_paciente")

    return paciente_mock


def verificar_interacoes_medicamentosas(medicamentos):
    print("🔧 Executando: verificar_interacoes_medicamentosas")

    if "ibuprofeno" in medicamentos:
        return {
            "risco": "moderado",
            "mensagem": "Possível atenção ao uso em paciente hipertenso."
        }

    return {
        "risco": "baixo",
        "mensagem": "Nenhuma interação relevante encontrada."
    }


def agendar_teleconsulta():
    print("🔧 Executando: agendar_teleconsulta")

    return {
        "status": "agendado",
        "data": "2026-05-20",
        "horario": "14:30"
    }

print("✅ Tools carregadas")


✅ Tools carregadas


## 7. Memória Conversacional

In [14]:
memoria = [
    {
        "role": "system",
        "content": system_prompt
    }
]

print("✅ Memória inicializada")


✅ Memória inicializada


## 8. Função principal do agente

In [15]:
from IPython.display import Markdown, display

def chamar_llm(mensagem_usuario):

    memoria.append({
        "role": "user",
        "content": mensagem_usuario
    })

    resposta = client.chat(
        model=MODEL_NAME,
        messages=memoria,
        options={
            "temperature": 0.3,
            "num_predict": 300
        },
        stream=False
    )

    texto = resposta["message"]["content"]

    memoria.append({
        "role": "assistant",
        "content": texto
    })

    display(Markdown(f"""
## 🤖 Resposta do BluaDiagnostics

{texto}
"""))

    return texto


## 9. Demonstração — Conversa Multi-turno

In [16]:
chamar_llm(
    "Olá, sou hipertenso e hoje acordei com dor de cabeça leve."
)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
Você relatou ser hipertenso e acordou hoje com dor de cabeça leve.

**2. Pontos de atenção**  
- Verifique se a dor de cabeça está acompanhada de: visão turva, náuseas/vômitos, tontura, confusão mental, fraqueza ou formigamento em algum membro.  
- Avalie a pressão arterial (PA) em casa, se possível, e anote o valor.  
- Observe se a dor piora ao mudar de posição, se há sensação de pulsação forte ou se surge após esforço físico.  
- Fique atento a sinais de alerta que exigem atenção imediata (ver item 4).

**3. Próxima ação**  
1


'**1. Resumo do caso**  \nVocê relatou ser hipertenso e acordou hoje com dor de cabeça leve.\n\n**2. Pontos de atenção**  \n- Verifique se a dor de cabeça está acompanhada de: visão turva, náuseas/vômitos, tontura, confusão mental, fraqueza ou formigamento em algum membro.  \n- Avalie a pressão arterial (PA) em casa, se possível, e anote o valor.  \n- Observe se a dor piora ao mudar de posição, se há sensação de pulsação forte ou se surge após esforço físico.  \n- Fique atento a sinais de alerta que exigem atenção imediata (ver item 4).\n\n**3. Próxima ação**  \n1'

In [17]:
chamar_llm(
    "Minha pressão ficou 135 por 85 e senti tontura ao levantar."
)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
- Hipertensão conhecida.  
- Pressão arterial medida em casa: 135 × 85 mmHg (dentro da faixa de controle para muitos pacientes, mas ainda acima da meta ideal <130/80 mmHg).  
- Relatou tontura ao levantar‑se (possível hipotensão ortostática).

**2. Pontos de atenção**  
- **Tontura ao mudar de posição**: pode indicar queda transitória da pressão arterial ao ficar em pé.  
- **Sintomas associados**: visão turva, sensação de desmaio, fraqueza, palpitações, dor no peito ou falta de ar.  
- **Frequência**: se a tontura ocorrer sempre que se levanta ou se tornar mais frequente.  
- **Uso de medicação antihipertensiva**: horário da última dose, se tomou a medicação antes de dormir ou logo ao acordar.  
- **Hidratação e ingestão de sódio**: baixa ingestão pode agravar a hipotensão ortostática.  
-


'**1. Resumo do caso**  \n- Hipertensão conhecida.  \n- Pressão arterial medida em casa: 135\u202f×\u202f85\u202fmmHg (dentro da faixa de controle para muitos pacientes, mas ainda acima da meta ideal <130/80\u202fmmHg).  \n- Relatou tontura ao levantar‑se (possível hipotensão ortostática).\n\n**2. Pontos de atenção**  \n- **Tontura ao mudar de posição**: pode indicar queda transitória da pressão arterial ao ficar em pé.  \n- **Sintomas associados**: visão turva, sensação de desmaio, fraqueza, palpitações, dor no peito ou falta de ar.  \n- **Frequência**: se a tontura ocorrer sempre que se levanta ou se tornar mais frequente.  \n- **Uso de medicação antihipertensiva**: horário da última dose, se tomou a medicação antes de dormir ou logo ao acordar.  \n- **Hidratação e ingestão de sódio**: baixa ingestão pode agravar a hipotensão ortostática.  \n-'

## 10. Demonstração de memória

In [18]:
print("📌 Histórico armazenado:\n")

for item in memoria:
    print(item["role"].upper())
    print(item["content"][:300])
    print("-" * 50)


📌 Histórico armazenado:

SYSTEM

Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Re
--------------------------------------------------
USER
Olá, sou hipertenso e hoje acordei com dor de cabeça leve.
--------------------------------------------------
ASSISTANT
**1. Resumo do caso**  
Você relatou ser hipertenso e acordou hoje com dor de cabeça leve.

**2. Pontos de atenção**  
- Verifique se a dor de cabeça está acompanhada de: visão turva, náuseas/vômitos, tontura, confusão mental, fraqueza ou formigamento em algum membro.  
- Avalie a pressão arterial (
--------------------------------------------------
USER
Minha pressão ficou 135 por 85 e senti tontura ao levantar.
--------------------------------------------------
ASSISTANT
**1. Resumo d

## 11. Demonstração de Tool Calling

In [19]:
historico = consultar_historico_paciente("PAC001")

print(json.dumps(
    historico,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: consultar_historico_paciente
{
  "paciente_id": "PAC001",
  "nome": "Paciente Simulado",
  "idade": 42,
  "condicoes": [
    "hipertensão leve"
  ],
  "medicamentos": [
    "losartana 50mg"
  ],
  "alergias": [
    "dipirona"
  ],
  "ultima_pressao": "135/85",
  "ultima_teleconsulta": "2026-05-10"
}


In [20]:
interacoes = verificar_interacoes_medicamentosas(
    ["losartana", "ibuprofeno"]
)

print(json.dumps(
    interacoes,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: verificar_interacoes_medicamentosas
{
  "risco": "moderado",
  "mensagem": "Possível atenção ao uso em paciente hipertenso."
}


In [21]:
teleconsulta = agendar_teleconsulta()

print(json.dumps(
    teleconsulta,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: agendar_teleconsulta
{
  "status": "agendado",
  "data": "2026-05-20",
  "horario": "14:30"
}


## 12. Resposta contextualizada usando tools

In [22]:
contexto = f'''
Histórico do paciente:
{historico}

Interações medicamentosas:
{interacoes}

Teleconsulta:
{teleconsulta}

Gere uma orientação segura para o paciente.
'''

chamar_llm(contexto)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
- Paciente de 42 anos com hipertensão leve, em uso de losartana 50 mg.  
- Medição de pressão arterial em casa: 135 × 85 mmHg.  
- Relata tontura ao levantar‑se (possível hipotensão ortostática).  
- Alergia conhecida a dipirona.  
- Teleconsulta já agendada para 20/05/2026 às 14:30.

**2. Pontos de atenção**  
- **Tontura ortostática**: verifique se ocorre sempre que muda de posição (sentado → em pé) e se melhora ao sentar ou deitar.  
- **Sintomas associados**: fique alerta a visão turva, sensação de desmaio, palpitações, dor no peito ou falta de ar. Caso algum desses apareça, procure atendimento imediato.  
- **Horário da medicação**: a losartana costuma ser tomada pela manhã; se estiver tomando à noite,


'**1. Resumo do caso**  \n- Paciente de 42\u202fanos com hipertensão leve, em uso de losartana 50\u202fmg.  \n- Medição de pressão arterial em casa: 135\u202f×\u202f85\u202fmmHg.  \n- Relata tontura ao levantar‑se (possível hipotensão ortostática).  \n- Alergia conhecida a dipirona.  \n- Teleconsulta já agendada para 20/05/2026 às 14:30.\n\n**2. Pontos de atenção**  \n- **Tontura ortostática**: verifique se ocorre sempre que muda de posição (sentado → em pé) e se melhora ao sentar ou deitar.  \n- **Sintomas associados**: fique alerta a visão turva, sensação de desmaio, palpitações, dor no peito ou falta de ar. Caso algum desses apareça, procure atendimento imediato.  \n- **Horário da medicação**: a losartana costuma ser tomada pela manhã; se estiver tomando à noite,'

# Conclusão

A PoC demonstrou:

- uso de Prompt Engineering;
- system prompt clínico;
- memória conversacional;
- function calling simulado;
- integração com Ollama Cloud;
- guardrails clínicos;
- fluxo Human-in-the-Loop.

O projeto atende os requisitos principais da Sprint 1 do Challenge BluaDiagnostics.
